In [22]:
import pandas as pd 
dataset=pd.read_csv('qoute_dataset.csv')
dataset.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
quote=dataset['quote']
import string
def function(text):
    text=text.lower()
    text=text.translate(str.maketrans('','',string.punctuation))
    text=text.split()

    return text 
tokens=[function(i) for i in quote]

vocabulary={"<pov>":0}
indx=1
for s in tokens:
    for i in s : 
        if i not in vocabulary:
            vocabulary[i]=indx 
            indx+=1 
embad=[]
for i in tokens:
    row=[vocabulary[s] for s in i]
    embad.append(row)



X=[]
y=[]
for sent in embad :
    for i in range(len(sent)):
        X.append(sent[:i]) # use this to collect till the last sequence 
        y.append(sent[i]) # only collect sequence one by one 


import torch
from torch.nn.utils.rnn import pad_sequence 
X=[torch.tensor(i,dtype=torch.long) for i in X]
y=torch.tensor(y,dtype=torch.long)
X_padded=pad_sequence(X,batch_first=True,padding_value=0)
X_padded=torch.tensor(X_padded,dtype=torch.long)

# Model Create 
class NXT_seq(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.embad=torch.nn.Embedding(len(vocabulary),64)
        self.LSTM=torch.nn.LSTM(64,32,batch_first=True),
        self.clasifier=torch.nn.Sequential(
            torch.nn.Linear(32,64),
            torch.nn.ReLU(),
            torch.nn.Linear(64,len(vocabulary))
        )
    def forward(self,x):
        x=self.embad(x)
        out,(h,c)=self.LSTM(x)
        x=out[:,-1,:]
        x=self.clasifier(x)
        return x 
Model=NXT_seq()

loss_fun=torch.nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=Model.parameters(),lr=0.001,weight_decay=0.01)


loss_list=[]
accuracy_li=[]
epochs=100
for epoch in range(epochs):
    output=Model(X_padded)
    loss=loss_fun(output,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # Predictions 
    pred=torch.argmax(output,dim=1)
    accuracy=(pred==y).float().mean()
    accuracy_li.append(accuracy.item())
    loss_list.append(loss.item())

    print(f'the loss is :{loss.item()} and the accuracy is :{accuracy.item()}')

/var/folders/j2/t5w7lcnj6wg9gmp3h6zxzlk00000gn/T/ipykernel_5195/722649392.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_padded=torch.tensor(X_padded,dtype=torch.long)


: 

In [ ]:
import matplotlib.pyplot as plt 
plt.plot(accuracy_li)
plt.plot(loss_list)
plt.style.use('dark_background')

torch.int64


In [ ]:
from torchinfo import summary 
summary(Model)